In [1]:
# Pin a Python-3.12-compatible Transformers 4.x environment.
# The original SparseGPT repository predates the current Transformers API.
!pip install -q --upgrade --only-binary=tokenizers \
    "transformers==4.57.6" \
    "tokenizers==0.22.1" \
    "datasets>=2.18,<5" \
    sentencepiece accelerate safetensors


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import sys
import torch
import transformers
import datasets

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU runtime before continuing: Runtime > Change runtime type > GPU")


Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.8.5
CUDA available: True
GPU: Tesla T4


In [3]:
# Make this cell safe to rerun.
!rm -rf /content/sparsegpt
!git clone https://github.com/IST-DASLab/sparsegpt.git /content/sparsegpt
%cd /content/sparsegpt


Cloning into '/content/sparsegpt'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 46 (delta 22), reused 10 (delta 10), pack-reused 14 (from 2)
Receiving objects: 100% (46/46), 26.80 KiB | 2.23 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/sparsegpt


In [4]:
from pathlib import Path

# Hugging Face moved WikiText to the Salesforce namespace.
path = Path("datautils.py")
text = path.read_text()
old = "load_dataset('wikitext',"
new = "load_dataset('Salesforce/wikitext',"
count = text.count(old)
assert count == 2, f"Expected 2 WikiText calls, found {count}. The upstream file may have changed."
path.write_text(text.replace(old, new))
print("Patched datautils.py for Salesforce/wikitext.")


Patched datautils.py for Salesforce/wikitext.


In [5]:
from pathlib import Path

path = Path("llama.py")
text = path.read_text()


def replace_exact(source, old, new, expected, label):
    count = source.count(old)
    assert count == expected, (
        f"{label}: expected {expected} occurrence(s), found {count}. "
        "The upstream SparseGPT file may have changed."
    )
    return source.replace(old, new)

# Modern Transformers decoder layers return a Tensor, while old versions returned a tuple.
# Normalize both cases and remove the batch dimension before assigning into outs[j].
text = replace_exact(
    text,
    "from quant import *\n",
    "from quant import *\n\n"
    "def _layer_hidden_states(output):\n"
    "    hidden = output[0] if isinstance(output, (tuple, list)) else output\n"
    "    return hidden[0]\n",
    1,
    "insert layer-output helper",
)

# The old code saved only attention_mask. Modern LLaMA layers also require
# position_ids, cache_position, and especially position_embeddings.
text = replace_exact(
    text,
    'cache = {"i": 0, "attention_mask": None}',
    'cache = {"i": 0, "layer_kwargs": None}',
    2,
    "replace cache dictionaries",
)
text = replace_exact(
    text,
    'cache["attention_mask"] = kwargs["attention_mask"]',
    'cache["layer_kwargs"] = dict(kwargs)',
    2,
    "capture complete decoder kwargs",
)
text = replace_exact(
    text,
    'attention_mask = cache["attention_mask"]',
    'layer_kwargs = cache["layer_kwargs"]',
    2,
    "restore complete decoder kwargs",
)
text = replace_exact(
    text,
    'outs[j] = layer(inps[j].unsqueeze(0), attention_mask=attention_mask)[0]',
    'outs[j] = _layer_hidden_states(layer(inps[j].unsqueeze(0), **layer_kwargs))',
    3,
    "update direct decoder-layer calls",
)

# The original loop always evaluates three datasets. Respect the CLI dataset instead.
text = replace_exact(
    text,
    'for dataset in ["wikitext2", "ptb", "c4"]:',
    'for dataset in [args.dataset]:',
    1,
    "restrict evaluation dataset",
)

path.write_text(text)
print("Patched llama.py for Transformers 4.57.x and WikiText-only evaluation.")


Patched llama.py for Transformers 4.57.x and WikiText-only evaluation.


In [6]:
# Quick static checks before downloading the model.
!python -m py_compile llama.py datautils.py sparsegpt.py modelutils.py quant.py
!grep -n "layer_kwargs\|_layer_hidden_states\|for dataset in" llama.py


10:def _layer_hidden_states(output):
50:    cache = {"i": 0, "layer_kwargs": None}
60:            cache["layer_kwargs"] = dict(kwargs)
77:    layer_kwargs = cache["layer_kwargs"]
122:                outs[j] = _layer_hidden_states(layer(inps[j].unsqueeze(0), **layer_kwargs))
140:            outs[j] = _layer_hidden_states(layer(inps[j].unsqueeze(0), **layer_kwargs))
172:    cache = {"i": 0, "layer_kwargs": None}
182:            cache["layer_kwargs"] = dict(kwargs)
199:    layer_kwargs = cache["layer_kwargs"]
215:            outs[j] = _layer_hidden_states(layer(inps[j].unsqueeze(0), **layer_kwargs))
334:    for dataset in [args.dataset]:


In [ ]:
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT = "/content/tinyllama-sparsegpt-50"

!python llama.py \
    {MODEL} \
    wikitext2 \
    --sparsity 0.5 \
    --nsamples 128 \
    --save {OUTPUT}


2026-07-17 16:52:13.375633: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
config.json: 100% 608/608 [00:00<00:00, 3.43MB/s]
model.safetensors: 100% 2.20G/2.20G [00:16<00:00, 135MB/s]
generation_config.json: 100% 124/124 [00:00<00:00, 1.25MB/s]
tokenizer_config.json: 1.29kB [00:00, 9.12MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 2.32MB/s]
special_tokens_map.json: 100% 551/551 [00:00<00:00, 5.72MB/s]
tokenizer.json: 1.84MB [00:00, 71.0MB/s]
README.md: 10.5kB [00:00, 47.9MB/s]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): 100% 733k/733k [00:00<00:00, 3.55MB/s]
wikitext-2-raw-v1/train-00000-of-00001.p(…): 100% 6.36M/6.36M [00:00<00:00, 24.1MB/s]
wikitext-2-raw-v1/validation-00000-of-00(…): 10

In [ ]:
# Save the tokenizer beside the pruned model so the output folder is directly reusable.
from transformers import AutoTokenizer
from pathlib import Path

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT = "/content/tinyllama-sparsegpt-50"

AutoTokenizer.from_pretrained(MODEL).save_pretrained(OUTPUT)
files = sorted(p.name for p in Path(OUTPUT).iterdir())
print("Saved files:")
for name in files:
    print(" -", name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Saved files:
 - chat_template.jinja
 - config.json
 - generation_config.json
 - model.safetensors
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer.model
 - tokenizer_config.json
